In [ ]:
merged_df = pd.read_csv('merged_df.csv')

# Getting ICD codes for breast, ovarian or colorectal cancers in diagnosis (SD) for each patient
icd_cancer_fx_prefixes = ['V16', 'Z80']
cancer_sd_fx = merged_df[merged_df['icd_code'].str.startswith(tuple(icd_cancer_fx_prefixes))]
cancer_sd_fx = cancer_sd_fx.reset_index(drop=True)

grouped_cancer_sd_fx_icd = cancer_sd_fx.groupby('subject_id', group_keys=False).apply(
    lambda x: ', '.join(sorted(set(x['icd_code'].astype(str))))
).reset_index(name='merged_icd_codes')


# Getting complete list of family history notes for patient
notes_df = pd.read_csv('notes_df.csv')
notes_df_unique = notes_df.drop_duplicates(subset=['subject_id', 'family_history'])

# Grouping different notes by patient
grouped_fx_notes = validated_d_all_unique.groupby('subject_id', group_keys=False).apply(
    lambda x: ' '.join(x['note_seq'].astype(str) + 'th note: ' + x['family_history'])
).reset_index(name='merged_family_history')

# Merging icd diagnosis and notes for each patient
fx_icd_and_notes = grouped_fx_notes.merge(grouped_cancer_sd_fx_icd, on='subject_id', how='left')

## Imputation of '___' based on the sentence structure, age and gender contexts using Azure OpenAI

# Adding Age and Gender for context
result_df = pd.merge(final_fx_icd_and_notes_valid, merged_df[['subject_id', 'gender',  'anchor_age']].drop_duplicates(subset='subject_id').reset_index(drop = True), on='subject_id', how='left')
result_df.shape

api_version = api_version
azure_endpoint = azure_endpoint
api_key = api_key
model = model


import openai
import pandas as pd

# Azure OpenAI configuration using the latest SDK
client = openai.AzureOpenAI(
    api_version = api_version,
    azure_endpoint = azure_endpoint,
    api_key = api_key
)

def impute_missing_values(text, gender, age):
    prompt = f"""
    You are an expert medical language model. Your task is to replace missing values marked as '___' or '__' or '____'
    in the provided family history text. Use the patient's gender ({gender}) and age ({age}) for context. 
    Ensure the imputed values are medically realistic and align with the overall context. Please do 
    not include any additional comments or notes during this task.

    Example Input:
    "Father, aged ___, suffers from Diverticular Disease for ___ years."
    Output:
    "Father, aged 67, suffers from Diverticular Disease for 10 years."

    Now process the following text:
    {text}

    Output:
    """
    
    try:
        response = client.chat.completions.create(
            model = model,   # Ensure this matches your Azure deployment name
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=500
        )
        
        # ✅ Added a check to avoid the 'NoneType' error
        content = response.choices[0].message.content
        if content is None:
            print("⚠️ Warning: No content returned in LLM response. Returning original text.")
            return text  # Return the original text if no content is generated
        else:
            return content.strip()
        
    except Exception as e:
        print(f"❌ Error during LLM call: {e}")
        return text  # Return the original text if an error occurs

# Apply imputation
result_df['imputed_family_history'] = result_df.apply(lambda row: impute_missing_values(
    row['merged_family_history'], row['gender'], row['anchor_age']), axis=1)

# Save the dataset
result_df.drop(['gender', 'anchor_age','merged_family_history'], axis=1, inplace=True)

result_df = result_df.rename(columns = {'imputed_family_history': 'family_history_notes',
                    'subject_id': 'id'})


# Removing any special characters - We will use this if needed, inactivated at this time
# result_df.family_history_notes = result_df.family_history_notes.replace("'","")
# result_df.family_history_notes = result_df.family_history_notes.replace('#','')
# result_df.family_history_notes = result_df.family_history_notes.replace('*','')
# result_df.family_history_notes = result_df.family_history_notes.replace('___','')

# Merging ICD codes with Notes
result_df['family_history_notes'] = '{' + result_df['merged_icd_codes'].astype(str) + '} ' + result_df['family_history_notes']

# Dropping ICD column
result_df.drop('merged_icd_codes', axis = 1, inplace = True)

# Dropping notes that are too long
result_df['length'] = result_df['family_history_notes'].str.len()
result_df.drop(result_df[result_df['length'] > 1500].index, inplace=True)
result_df.drop(['length', 'merged_icd_codes'], axis = 1, inplace = True)

# Saving the dataset
result_df.to_csv("result_df_input_for_ner_summary.csv", index=False)

In [ ]:
import re
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [ ]:
merged_df = pd.read_csv('merged_df.csv')
merged_df.columns

In [ ]:
pd.options.display.max_columns = None
merged_df.head()

### Getting ICD codes for breast, ovarian or colorectal cancers in diagnosis (SD) for each patient

In [ ]:
# icd_cancer_fx = ['V163', 'Z803', 'V1641', 'Z8041', 'V160', 'Z800']
# icd_cancer_fx_revised = 
# cancer_sd_fx = merged_df[merged_df['icd_code'].isin(icd_cancer_fx)]
# cancer_sd_fx = cancer_sd_fx.reset_index(drop = True)
# cancer_sd_fx.shape

icd_cancer_fx_prefixes = ['V16', 'Z80']
cancer_sd_fx = merged_df[merged_df['icd_code'].str.startswith(tuple(icd_cancer_fx_prefixes))]
cancer_sd_fx = cancer_sd_fx.reset_index(drop=True)
cancer_sd_fx.shape


In [ ]:
# icd_cancer_fx = ['V163', 'Z803', 'V1641', 'Z8041', 'V160', 'Z800']
# icd_cancer_fx_revised = 
# cancer_sd_fx = merged_df[merged_df['icd_code'].isin(icd_cancer_fx)]
# cancer_sd_fx = cancer_sd_fx.reset_index(drop = True)
# cancer_sd_fx.shape

icd_cancer_fx_prefixes = ['V16', 'Z80']
cancer_sd_fx = merged_df[merged_df['icd_code'].str.startswith(tuple(icd_cancer_fx_prefixes))]
cancer_sd_fx = cancer_sd_fx.reset_index(drop=True)
cancer_sd_fx.shape


In [ ]:
grouped_cancer_sd_fx_icd = cancer_sd_fx.groupby('subject_id', group_keys=False).apply(
    lambda x: ', '.join(sorted(set(x['icd_code'].astype(str))))
).reset_index(name='merged_icd_codes')

In [ ]:
grouped_cancer_sd_fx_icd.shape

In [ ]:
grouped_cancer_sd_fx_icd.shape

In [ ]:
grouped_cancer_sd_fx_icd.head()

### Getting complete list of family history notes for patient

In [ ]:
validated_d_all = pd.read_csv('validated_d_all.csv')
validated_d_all.drop('text', axis = 1, inplace = True)
print('Total number of notes: ', validated_d_all.shape[0])
validated_d_all_unique = validated_d_all.drop_duplicates(subset=['subject_id', 'family_history'])

print('Total number of unique notes: ', validated_d_all_unique.shape[0])

In [ ]:
grouped_fx_notes = validated_d_all_unique.groupby('subject_id', group_keys=False).apply(
    lambda x: ' '.join(x['note_seq'].astype(str) + 'th note: ' + x['family_history'])
).reset_index(name='merged_family_history')

In [ ]:
grouped_fx_notes.head()

### Merging icd diagnosis and notes for each patient

In [ ]:
fx_icd_and_notes = grouped_fx_notes.merge(grouped_cancer_sd_fx_icd, on='subject_id', how='left')

In [ ]:
fx_icd_and_notes.head()

### Merging fx_icd_and_notes with validated ground truths

In [ ]:
valid_fx = pd.read_csv('family_history_validated_100_copy_unopened.csv')
valid_fx = valid_fx.rename(columns={'id': 'note_id'})
valid_fx.drop('Unnamed: 0', axis = 1, inplace = True)
valid_fx = valid_fx.merge(validated_d_all[['note_id', 'subject_id']], on='note_id', how='left')
valid_fx.shape

In [ ]:
valid_fx.head()

### Possible that some notes for the same patient may be related to multiple cancers. Manucally checking

In [ ]:
duplicates_df = valid_fx[valid_fx.duplicated(subset='subject_id', keep=False)]
print('Total number of patients with more than one note:', duplicates_df.subject_id.nunique())

In [ ]:
duplicates_df.sort_values(by = 'subject_id')

Patient with conflicting validations: (subject ID redacted for public release)

In [ ]:
valid_fx[valid_fx['subject_id'] == CONFLICT_SUBJECT_ID  # value removed for public release; set from your local MIMIC data]

In [ ]:
valid_fx.loc[(valid_fx.index == 135) & (valid_fx['subject_id'] == CONFLICT_SUBJECT_ID  # value removed for public release; set from your local MIMIC data), 'ground_truth_breast_cancer'] = 1

#### Final list of unique ground truths

In [ ]:
valid_fx = valid_fx.drop_duplicates(subset='subject_id')
valid_fx.drop('note_id', axis = 1, inplace = True)

### Merging fx_icd_and_notes with validated ground truths

In [ ]:
final_fx_icd_and_notes_valid = fx_icd_and_notes.merge(valid_fx, on='subject_id', how='left')

In [ ]:
final_fx_icd_and_notes_valid.columns

In [ ]:
final_fx_icd_and_notes_valid.head()

In [ ]:
final_fx_icd_and_notes_valid = final_fx_icd_and_notes_valid[['subject_id', 
                                                             'merged_family_history', 
                                                             'merged_icd_codes',
                                                             'ground_truth_breast_cancer',
                                                             'ground_truth_ovarian_cancer',
                                                             'ground_truth_colorectal_cancer']]

In [ ]:
final_fx_icd_and_notes_valid.head()

In [ ]:
final_fx_icd_and_notes_valid.shape

### Imputation of '___' based on the sentence structure, age and gender contexts using Azure OpenAI

#### Adding Age and Gender for context

In [ ]:
result_df = pd.merge(final_fx_icd_and_notes_valid, merged_df[['subject_id', 'gender',  'anchor_age']].drop_duplicates(subset='subject_id').reset_index(drop = True), on='subject_id', how='left')
result_df.shape

In [ ]:
result_df.head()

In [ ]:
result_df.info()

In [ ]:
api_version = '2024-02-15-preview'
azure_endpoint = ''
api_key = ''
model = ''

In [ ]:
import openai
import pandas as pd

# Azure OpenAI configuration using the latest SDK
client = openai.AzureOpenAI(
    api_version = api_version,
    azure_endpoint = azure_endpoint,
    api_key = api_key
)

def impute_missing_values(text, gender, age):
    prompt = f"""
    You are an expert medical language model. Your task is to replace missing values marked as '___' 
    in the provided family history text. Use the patient's gender ({gender}) and age ({age}) for context. 
    Ensure the imputed values are medically realistic and align with the overall context. Please do 
    not include any additional comments or notes during this task.

    Example Input:
    "Father, aged ___, suffers from Diverticular Disease for ___ years."
    Output:
    "Father, aged 67, suffers from Diverticular Disease for 10 years."

    Now process the following text:
    {text}

    Output:
    """
    
    try:
        response = client.chat.completions.create(
            model = model,   # Ensure this matches your Azure deployment name
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=500
        )
        
        # ✅ Added a check to avoid the 'NoneType' error
        content = response.choices[0].message.content
        if content is None:
            print("⚠️ Warning: No content returned in LLM response. Returning original text.")
            return text  # Return the original text if no content is generated
        else:
            return content.strip()
        
    except Exception as e:
        print(f"❌ Error during LLM call: {e}")
        return text  # Return the original text if an error occurs

# Read dataset
df = result_df.copy()

# Apply imputation
df['imputed_family_history'] = df.apply(lambda row: impute_missing_values(
    row['merged_family_history'], row['gender'], row['anchor_age']), axis=1)

# Save the dataset
df.drop(['gender', 'anchor_age','merged_family_history'], axis=1, inplace=True)
df = df[['subject_id', 'imputed_family_history', 'merged_icd_codes', 
         'ground_truth_breast_cancer', 'ground_truth_ovarian_cancer', 'ground_truth_colorectal_cancer']]

In [ ]:
pd.set_option('display.max_colwidth', None)
df.head()

In [ ]:
a = df.copy()

In [ ]:
def clean_text(text):
    # Remove any initial lines that appear to be introductory or system-generated
    text = re.sub(r"^(.*?values.*?:|.*?with medically realistic.*?:|.*?provided text.*?:|.*?Output.*?:|.*?Completed.*?:)", "", text, flags=re.IGNORECASE | re.DOTALL)

    # Optionally, remove blank lines at the start if they appear after removal
    text = re.sub(r"^\s*\n", "", text)

    # Strip leading and trailing whitespace
    return text.strip()

# Apply cleaning to the column
df['imputed_family_history'] = df['imputed_family_history'].apply(clean_text)

In [ ]:
df = df.rename(columns = {'imputed_family_history': 'family_history_notes',
                    'subject_id': 'id'})

In [ ]:
df.family_history_notes = df.family_history_notes.replace("'","")
# df.family_history_notes = df.family_history_notes.replace('"','')
df.family_history_notes = df.family_history_notes.replace('#','')
df.family_history_notes = df.family_history_notes.replace('___','')

In [ ]:
df.head(30)

In [ ]:
df['family_history_notes'] = '{' + df['merged_icd_codes'].astype(str) + '} ' + df['family_history_notes']

In [ ]:
df.drop('merged_icd_codes', axis = 1, inplace = True)

In [ ]:
df.drop('merged_icd_codes', axis = 1, inplace = True)

df['length'] = df['family_history_notes'].str.len()

df.drop(df[df['length'] > 1500].index, inplace=True)
df.drop(['length', 'merged_icd_codes'], axis = 1, inplace = True)

df.to_csv("result_df_output_final_with_all_cancer_SDs.csv", index=False)

In [ ]:
df.drop(df[df['length'] > 1500].index, inplace=True)
df.drop(['length', 'merged_icd_codes'], axis = 1, inplace = True)

In [ ]:
df['length'].describe()

In [ ]:
df[df['family_history_notes'].str.len() > 2000].shape

In [ ]:
df.to_csv("result_df_output_final_with_all_cancer_SDs.csv", index=False)

In [ ]:
d = df.copy()

In [ ]:
replacements = {
    'V163': 'CANCER, BREAST',
    'Z803': 'CANCER, BREAST',
    'V1641': 'CANCER, OVARIAN',
    'Z8041': 'CANCER, OVARIAN',
    'V160': 'CANCER, COLON',
    'Z800': 'CANCER, COLON',
    'nan': 'OTHER'
}

d['family_history_notes'] = d['family_history_notes'].replace(replacements, regex=True)

In [ ]:
df.shape

In [ ]:
d.head()

In [ ]:
merged_df = pd.read_csv('merged_df.csv')
merged_df.columns

In [ ]:
merged_df.shape